# Firmar un PDF y dejar constancia visible: la medición

Cuaderno de lectura de la medición `firma-pdf/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos): tres páginas del NIST CSF 2.0 firmadas en PAdES con un pie visible y un código QR en cada página, sin sello y con el sello cualificado de Sectigo (Europe) SL, por un firmante ficticio de una autoridad de pruebas; la legibilidad del QR con ZXing; y los servicios de sellado de tiempo cualificado de la lista de confianza española. Los ficheros de `datos/` los escribe `generar.py` con la copia de la aplicación que hay en `herramienta/`. Este cuaderno solo los lee: no firma ni consulta ningún servicio.

In [ ]:
import json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/firma-pdf/datos/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path("datos") / nombre
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def json_de(nombre):
    p = Path("datos") / nombre
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return json.loads(r.read().decode("utf-8"))

R = json_de("resumen.json")
print("medición del", R["fecha"][:10], "·", R["documento"]["paginas_extracto"], "páginas del NIST CSF 2.0")

## El pie: espacio y peso

In [ ]:
G = R["geometria"]
pags = R["documento"]["paginas_extracto"]
print("banda del pie:", G["banda_pt"], "pt =", G["banda_mm"], "mm · QR de", G["lado_qr_mm"], "mm")
print("reducción del contenido en A4:", round(100 * (1 - G["factor_a4"]), 1), "%")
print("kB por página con el pie:", round((G["bytes_con_pie_sin_firma"] - G["bytes_documento"]) / pags / 1000, 1),
      "· sin recomprimir:", round((G["bytes_con_pie_sin_recomprimir"] - G["bytes_documento"]) / pags / 1000, 1))

## La firma, sin sello y con el sello cualificado

In [ ]:
F = R["firmas"]
filas = []
for clave, nombre in (("sin_sello", "sin sello"), ("sello_cualificado", "sello cualificado")):
    f = F[clave]
    filas.append({"firma": nombre, "repeticiones": f["repeticiones"],
                  "mediana_s": f["segundos_mediana"], "bytes": f["bytes"],
                  **{f"valida_{k}": v for k, v in f["validacion"].items()},
                  "intacta_con_un_byte_alterado": f["validacion_con_un_byte_alterado"]["intacta"]})
pd.DataFrame(filas).set_index("firma").T

## Legibilidad del QR con ZXing

In [ ]:
qr = leer("qr_legibilidad.csv")
def umbral(g):
    g = g.sort_values("ppp"); ok = g.leido.to_numpy()
    return next((int(g.ppp.iloc[i]) for i in range(len(ok)) if ok[i:].all()), None)
resumen = qr.groupby("contenido").apply(lambda g: pd.Series({
    "bytes": int(g.bytes.iloc[0]), "modulos": int(g.modulos.iloc[0]),
    "mm_por_modulo": g.mm_por_modulo.iloc[0], "lee_desde_ppp": umbral(g)}), include_groups=False)
print(resumen.sort_values("bytes").to_string())
tabla = qr.pivot(index="contenido", columns="ppp", values="leido").loc[resumen.sort_values("bytes").index]
plt.imshow(tabla, cmap="RdYlGn", aspect="auto"); plt.yticks(range(len(tabla)), tabla.index); plt.xticks(range(len(tabla.columns)), tabla.columns)
plt.xlabel("ppp"); plt.tight_layout(); plt.show()

## Sellado de tiempo cualificado en la lista de confianza española

In [ ]:
tsl = leer("tsl_sellado_cualificado.csv")
T = R["tsl"]
print("edición de la lista:", T["emision"][:10], "· servicios TSA/QTST:", len(tsl))
print(tsl.estado.value_counts().sort_index().sort_values(ascending=False, kind="stable").to_dict())
conc = tsl[tsl.estado == "granted"]
print("prestadores con alguno concedido:", conc.prestador.nunique())
print(tsl[tsl.prestador.str.contains("Sectigo")][["prestador", "estado", "desde"]].to_string(index=False))

## Firmas del PDF del ENS en el BOE

In [ ]:
B = R["boe"]
print("formato de firma:", B["subfiltros"], "· firmas que lee pyHanko:", B["firmas_que_lee_pyhanko"])